Construction d’un CSV propre du taux de chômage (INSEE)

Données sources
INSEE – séries longues du taux de chômage au sens du BIT
Données trimestrielles par département depuis 1982

Script 1 — Cleaning et agrégation
    Suppression des valeurs manquantes (NaN)
    Virer département → garder que région
    Ajout des codes régions INSEE pour faciliter les merges et requêtes
    Agrégation temporelle : calcul de la moyenne annuelle du taux de chômage à partir des 4 trimestres
    Résultat : base annuelle du chômage par région.

Script 2 — Mise en forme (wide → long)
    Inversion de la matrice pour obtenir un format tidy
    Structure finale :
        année
        région
        code_région
        taux_chômage

Script 3 — Corrélation chômage / créations d’entreprises
    Fusion avec les données de créations d’entreprises, et export du fichier correlation_chomage_creation.csv pour tester une regression

In [1]:
import pandas as pd

# Charger le CSV
df = pd.read_csv("/Users/tristan/info/ENSAE/2A/python_DS_2A/data/data_outdated/valeurs_trimestrielles.csv", sep=';', encoding='utf-8')  # adapter le séparateur

# Supprimer les lignes où 'Libellé' contient 'Codes'
df = df[~df['Libellé'].str.contains('Codes', na=False)]

# Garder seulement les colonnes de 2015-T1 à 2025-T2 + la colonne Libellé
colonnes_a_garder = ['Libellé'] + [col for col in df.columns if col.startswith(tuple(str(y) for y in range(2015, 2026)))]
df = df[colonnes_a_garder]

# mettre des floats (vs obj)
for col in df.columns:
    if col != "Libellé":
        df[col] = pd.to_numeric(df[col], errors='raise')

    
# Créer de nouvelles colonnes pour les moyennes annuelles
annees = range(2015, 2026)
trimestres = ['T1', 'T2', 'T3', 'T4']

for annee in annees:
    colonnes_annee = [f"{annee}-{t}" for t in trimestres]
    # S'assurer que toutes les colonnes trimestrielles existent avant de calculer la moyenne
    colonnes_existantes = [col for col in colonnes_annee if col in df.columns]
    if colonnes_existantes:
        df[f"Moyenne_{annee}"] = df[colonnes_existantes].mean(axis=1)
    else:
        # Si aucune colonne trimestrielle n'existe pour l'année, la moyenne est NaN
        df[f"Moyenne_{annee}"] = float('nan')

# Supprimer les colonnes trimestrielles
colonnes_trimestres = [col for col in df.columns if '-' in col]  # ex: 2015-T1, 2024-T3
df = df.drop(columns=colonnes_trimestres)


#sauvergarde
df = df.round(3)        # arrondissement pour les floats trop longs (999999999 ou 000000001)

In [3]:
chomage=df

chomage = chomage.head(15)
chomage['region_nom'] = (
    chomage['Libellé']
    .str.replace('Taux de chômage localisé par région - ', '', regex=False)
    .str.strip()
)
#renommer la colonne Taux de chômage par région par region_nom

/var/folders/tm/96qft415759d_3psw39j35zc0000gn/T/ipykernel_19433/3173811411.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chomage['region_nom'] = (


In [4]:
import re

# -----------------------------------------------------------------------

# -----------------------------------------------------------------------
# 2. Transformation (Pivot -> Unpivot)

chomage1 = chomage.melt(
    id_vars=['region_nom'], 
    var_name='colonne_temporaire', 
    value_name='taux_chomage'
)

# Supprimer les taux manquants
chomage1 = chomage1.dropna(subset=['taux_chomage'])

# Extraction de l'année (peut créer des NaN)
chomage1['TIME_VALUE'] = chomage1['colonne_temporaire'].str.extract(r'(\d{4})')

# Supprimer les lignes sans année
chomage1 = chomage1.dropna(subset=['TIME_VALUE'])

# Conversion sûre
chomage1['TIME_VALUE'] = chomage1['TIME_VALUE'].astype(int)

# -----------------------------------------------------------------------
# 3. Nettoyage final

chomage_format_long = (
    chomage1
    .drop(columns='colonne_temporaire')
    .rename(columns={'region_nom': 'Region'})
    .rename(columns={'taux_chomage': 'Taux de chômage par région'})
    .sort_values(by=['Region', 'TIME_VALUE'])
    .reset_index(drop=True)
)

# Sauvegarde
chomage_format_long.to_csv("/Users/tristan/info/ENSAE/2A/python_DS_2A/data/Variables explicatives/chomage_format_long.csv")


In [5]:
creation_per_1000 = pd.read_csv("/Users/tristan/info/ENSAE/2A/python_DS_2A/data/Variables explicatives/creation_per_1000.csv") 
#colonne TIME_PERIOD en TIME_VALUE
creation_per_1000 = creation_per_1000.rename(columns={'TIME_PERIOD': 'TIME_VALUE'})
chomage_format_long = chomage_format_long.rename(columns={'Region': 'region_nom'})

In [6]:
df_merged = pd.merge(chomage_format_long, creation_per_1000, on=['region_nom', 'TIME_VALUE'], how='inner')

#envoie du df_merged vers le csv nommé 'correlation_chomage_creation.csv'
df_merged.to_csv("/Users/tristan/info/ENSAE/2A/python_DS_2A/data/Variables explicatives/correlation_chomage_creation.csv", index=False)

correlation_chomage_creation=df_merged